# NEPSE Stock Data Exploration

**Date**: October 22, 2025  
**Purpose**: Explore NEPSE stock price data for LSTM model development

This notebook explores the raw stock data to understand:
- Data structure and format
- Missing values and data quality
- Price trends and patterns
- Volume analysis
- Feature distributions

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Path to raw data
data_path = "../data/raw"

# List all stock files
stock_files = sorted([f for f in os.listdir(data_path) if f.endswith('.csv')])

print(f"Total number of stocks: {len(stock_files)}")
print(f"\nFirst 10 stocks:")
for i, file in enumerate(stock_files[:10], 1):
    stock_name = file.split('_')[0]
    print(f"  {i:2d}. {stock_name}")
    
print(f"\n... and {len(stock_files) - 10} more stocks")

In [ ]:
# Load NABIL stock data
sample_stock = "NABIL_2000-01-01_2021-12-31.csv"
df = pd.read_csv(os.path.join(data_path, sample_stock))

print(f"Loaded: {sample_stock}")
print(f"Shape: {df.shape} (rows, columns)")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# Statistical summary
print("=" * 70)
print("STATISTICAL SUMMARY")
print("=" * 70)
df.describe()

In [ ]:
# Plot OHLC (Open, High, Low, Close) - using Max/Min as approximations
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Max Price (High)
axes[0, 0].plot(df['Date'], df['Max. Price'], color='green', alpha=0.7)
axes[0, 0].set_title('Max Price (High)', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Price (NPR)')
axes[0, 0].grid(True, alpha=0.3)

# Min Price (Low)
axes[0, 1].plot(df['Date'], df['Min. Price'], color='red', alpha=0.7)
axes[0, 1].set_title('Min Price (Low)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Price (NPR)')
axes[0, 1].grid(True, alpha=0.3)

# Close Price
axes[1, 0].plot(df['Date'], df['Close Price'], color='blue', alpha=0.7)
axes[1, 0].set_title('Close Price', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Price (NPR)')
axes[1, 0].grid(True, alpha=0.3)

# Total Traded Shares (Volume)
axes[1, 1].plot(df['Date'], df['Total Traded Shares'], color='purple', alpha=0.7)
axes[1, 1].set_title('Trading Volume', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Date')
axes[1, 1].set_ylabel('Shares')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot price with moving averages
plt.figure(figsize=(16, 8))
plt.plot(df['Date'], df['Close Price'], label='Close Price', linewidth=2, alpha=0.8)
plt.plot(df['Date'], df['SMA_5'], label='SMA 5', linewidth=1.5, alpha=0.7)
plt.plot(df['Date'], df['SMA_10'], label='SMA 10', linewidth=1.5, alpha=0.7)
plt.plot(df['Date'], df['SMA_20'], label='SMA 20', linewidth=1.5, alpha=0.7)
plt.plot(df['Date'], df['SMA_50'], label='SMA 50', linewidth=1.5, alpha=0.7)

plt.title('NABIL Stock - Price with Moving Averages', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Price (NPR)', fontsize=12)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution plots
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Close Price distribution
axes[0, 0].hist(df['Close Price'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Close Price Distribution', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Price (NPR)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].grid(True, alpha=0.3)

# Daily Returns distribution
axes[0, 1].hist(df['Daily_Return'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[0, 1].set_title('Daily Returns Distribution', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Return (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# Volume distribution
axes[1, 0].hist(df['Total Traded Shares'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 0].set_title('Trading Volume Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Shares')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].grid(True, alpha=0.3)

# Volatility distribution
axes[1, 1].hist(df['Volatility_10'].dropna(), bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1, 1].set_title('Volatility Distribution (10-day)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Std Dev')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Load multiple stocks for comparison
sample_stocks = ['NABIL_2000-01-01_2021-12-31.csv', 
                 'ADBL_2000-01-01_2021-12-31.csv',
                 'EBL_2000-01-01_2021-12-31.csv',
                 'SCB_2000-01-01_2021-12-31.csv']

plt.figure(figsize=(16, 8))

for stock_file in sample_stocks:
    try:
        stock_name = stock_file.split('_')[0]
        stock_df = pd.read_csv(os.path.join(data_path, stock_file))
        stock_df['Date'] = pd.to_datetime(stock_df['Date'])
        stock_df = stock_df.sort_values('Date')
        
        # Normalize prices to start at 100 for comparison
        normalized_price = (stock_df['Close Price'] / stock_df['Close Price'].iloc[0]) * 100
        
        plt.plot(stock_df['Date'], normalized_price, label=stock_name, linewidth=1.5, alpha=0.8)
    except Exception as e:
        print(f"Could not load {stock_name}: {e}")

plt.title('Normalized Stock Price Comparison (Base = 100)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Normalized Price', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Key Observations:

1. **Data Quality**: 
   - Check if there are missing values
   - Verify date continuity (trading days vs calendar days)
   - Look for outliers in price/volume

2. **Price Trends**:
   - Long-term trends visible in moving averages
   - Volatility varies over time
   - Volume spikes may correlate with price movements

3. **Features for LSTM**:
   - Close Price (target variable)
   - Max/Min Price (daily range)
   - Volume (trading activity)
   - Moving averages (trend indicators)
   - Daily returns (momentum)
   - Volatility (risk measure)

4. **Data Preprocessing Needed**:
   - Normalize/standardize features
   - Handle missing values
   - Create sequences (lookback window)
   - Train/validation/test split

### Next Steps:

1. **Data Preprocessing**:
   - Create preprocessing script
   - Add more technical indicators (RSI, MACD, Bollinger Bands)
   - Save processed data

2. **PyTorch Dataset**:
   - Implement StockDataset class
   - Create sequences for LSTM input
   - Setup DataLoader

3. **Model Development**:
   - Design LSTM architecture
   - Implement training loop
   - Add evaluation metrics

4. **Training**:
   - Start with single stock
   - Tune hyperparameters
   - Scale to multiple stocks

## 10. Key Findings & Next Steps

## 9. Compare Multiple Stocks

In [ ]:
# Select numeric columns for correlation
numeric_cols = ['Total Transactions', 'Total Traded Shares', 'Total Traded Amount',
                'Max. Price', 'Min. Price', 'Close Price']

# Calculate correlation matrix
correlation = df[numeric_cols].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, fmt='.2f')
plt.title('Correlation Heatmap - Stock Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

## 7. Distribution Analysis

In [ ]:
# Calculate moving averages
df['SMA_5'] = df['Close Price'].rolling(window=5).mean()
df['SMA_10'] = df['Close Price'].rolling(window=10).mean()
df['SMA_20'] = df['Close Price'].rolling(window=20).mean()
df['SMA_50'] = df['Close Price'].rolling(window=50).mean()

# Calculate daily returns
df['Daily_Return'] = df['Close Price'].pct_change()

# Calculate volatility (standard deviation)
df['Volatility_10'] = df['Close Price'].rolling(window=10).std()

print("Technical indicators calculated!")
print(f"\nSample data with indicators:")
df[['Date', 'Close Price', 'SMA_5', 'SMA_20', 'Daily_Return', 'Volatility_10']].tail(10)

## 6. Calculate Basic Technical Indicators

In [ ]:
# Plot closing price over time
plt.figure(figsize=(16, 6))
plt.plot(df['Date'], df['Close Price'], linewidth=1.5, alpha=0.8)
plt.title('NABIL Stock - Closing Price Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Close Price (NPR)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')

print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Total trading days: {len(df)}")

## 5. Visualize Stock Price Trends

In [ ]:
# Check for missing values
print("=" * 70)
print("MISSING VALUES")
print("=" * 70)
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

In [ ]:
# Data info
print("=" * 70)
print("DATA INFORMATION")
print("=" * 70)
df.info()

print("\n" + "=" * 70)
print("COLUMN NAMES")
print("=" * 70)
print(df.columns.tolist())

## 4. Data Info and Statistics

## 3. Load Sample Stock Data (NABIL)

## 2. Explore Available Data

## 1. Import Libraries